# Task 4 Apply an Advanced AI Technique

I would use MLflow experiment tracking to record the regression models developed in Tasks 1 and 3.

I would record their settings, validation and test metrics, and saved artifacts. 

Tracking helps in comparing models and retain their execution evidence. It does not improve prediction accuracy by itself.

In [1]:
# Configure Logging

import logging
from pathlib import Path
from importlib.util import find_spec

import pandas as pd

PART3_DIR = Path.cwd().parent
REPO_DIR = PART3_DIR.parent

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            PART3_DIR / "part3_smart_city_capstone_advanced_ai.log",
            encoding="utf-8"
        )
    ],
    force=True
)


logger = logging.getLogger(__name__)

logger.info("Advanced AI experiment tracking notebook has started.")

mlflow_available = find_spec("mlflow") is not None

display(pd.DataFrame([{
    "package": "mlflow",
    "available": mlflow_available
}]))


logger.info(
    "MLflow availability check has completed: %s.",
    mlflow_available
)

2026-10-11 09:36:17,055 | INFO | __main__ | Advanced AI experiment tracking notebook has started.


,package,available
0,mlflow,True


2026-10-11 09:36:17,105 | INFO | __main__ | MLflow availability check has completed: True.


## Verify MLflow

I would check that MLflow imports successfully and record its version before configuring the local tracking store.

In [2]:
# Verify the installed MLflow package.

import mlflow

logger.info(
    "MLflow %s imported successfully.",
    mlflow.__version__
)


display(pd.DataFrame([{
    "package": "mlflow",
    "version": mlflow.__version__
}]))

2026-10-11 09:37:33,210 | INFO | __main__ | MLflow 3.17.0 imported successfully.


,package,version
0,mlflow,3.17.0


## Configure local experiment tracking

I wouild store experiment metadata in a local SQLite database and model artifacts in a separate folder under Part 3.

I woudl use one named experiment to compare the existing regression results. 

In [3]:
# Configure the local tracking database and artifact folder.

tracking_dir = PART3_DIR / "mlflow"
artifacts_dir = tracking_dir / "artifacts"

artifacts_dir.mkdir(parents=True, exist_ok=True)

database_path = tracking_dir / "tracking.db"

tracking_uri = (
    "sqlite:///" + database_path.resolve().as_posix()
)

mlflow.set_tracking_uri(tracking_uri)

experiment_name = "Smart City Traffic Regression"

experiment = mlflow.get_experiment_by_name(experiment_name)


if experiment is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location="file:../mlflow/artifacts"
    )
else:
    experiment_id = experiment.experiment_id

# Keep persisted artifact paths relative to the notebooks folder.
import sqlite3

with sqlite3.connect(database_path) as connection:
    connection.execute(
        "UPDATE experiments SET artifact_location=? WHERE experiment_id=?",
        ("file:../mlflow/artifacts", experiment_id)
    )
    connection.execute(
        "UPDATE experiments SET artifact_location=? WHERE experiment_id=0",
        ("file:../mlflow/artifacts/default/0",)
    )
    run_ids = connection.execute(
        "SELECT run_uuid FROM runs WHERE experiment_id=?",
        (experiment_id,)
    ).fetchall()
    for (run_id,) in run_ids:
        connection.execute(
            "UPDATE runs SET artifact_uri=? WHERE run_uuid=?",
            (f"file:../mlflow/artifacts/{run_id}/artifacts", run_id)
        )

mlflow.set_experiment(experiment_name)

logger.info(
    "Local MLflow tracking is configured for experiment %s.",
    experiment_name
)


display(pd.DataFrame([{
    "experiment": experiment_name,
    "experiment_id": experiment_id,
    "metadata_store": "mlflow/tracking.db",
    "artifact_folder": "mlflow/artifacts"
}]))



2026/10/11 09:39:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/11 09:39:30 INFO mlflow.store.db.utils: Updating database tables
2026-10-11 09:39:33,504 | INFO | __main__ | Local MLflow tracking is configured for experiment Smart City Traffic Regression.


,experiment,experiment_id,metadata_store,artifact_folder
0,Smart City Traffic Regression,1,mlflow/tracking.db,mlflow/artifacts


## Load the existing model results

I would load the validation and test results from Tak 3, together with the saved Random Forest and neural network settings.

These are previously completed evaluations.

In [4]:
#Load saved results and model settings for experiment tracking.

import joblib
import numpy as np

results_dir = PART3_DIR / "results"
models_dir = PART3_DIR / "models"
figures_dir = PART3_DIR / "figures"

validation_results = pd.read_csv(
    results_dir / "task3_validation_comparison.csv"
)


test_results = pd.read_csv(
    results_dir / "task3_test_comparison.csv"
)

model_results = validation_results.merge(
    test_results,
    on="model",
    validate="one_to_one"
)


expected_models = {"Neural Network", "Random Forest"}

assert set(model_results["model"]) == expected_models

metric_columns = [
    "validation_MAE",
    "validation_R_squared",
    "test_MAE",
    "test_R_squared"
]

assert np.isfinite(
    model_results[metric_columns].to_numpy()
).all()

# Read actual settings from the saved artifacts.
saved_forest = joblib.load(
    models_dir / "random_forest_regression_v1.joblib"
)


forest_parameters = (
    saved_forest.named_steps["model"].get_params()
)

neural_config = joblib.load(
    models_dir / "neural_network_config_v1.joblib"
)

# Confirm that the artifacts needed for both runs are available.
required_artifacts = [
    models_dir / "random_forest_regression_v1.joblib",
    models_dir / "preparation_config_v1.joblib",
    models_dir / "neural_network_traffic_v1.keras",
    models_dir / "neural_network_preprocessor_v1.joblib",
    models_dir / "neural_network_config_v1.joblib",
    results_dir / "neural_network_training_history.csv",
    figures_dir / "neural_network_training_curve.png",
    figures_dir / "shap_global_importance.png",
    figures_dir / "shap_local_waterfall.png"
]


for artifact in required_artifacts:
    if not artifact.is_file():
        raise FileNotFoundError(
            f"Required artifact is missing: {artifact.name}."
        )


logger.info(
    "Loaded results for %s models and verified the required artifacts.",
    len(model_results)
)



display(model_results.round(4))

2026-10-11 09:42:30,460 | INFO | __main__ | Loaded results for 2 models and verified the required artifacts.


,model,validation_MAE,validation_R_squared,test_MAE,test_R_squared
0,Neural Network,316.4414,0.9307,256.9366,0.9455
1,Random Forest,317.5692,0.9256,250.8358,0.9448


## Record model runs in MLflow

I woudl create one run for Random Forest and one for the neural network.

Each run  would recor the model version, training settings, chronological split sizes, validation and tst metrics, and saved artifacts.

These runs record previously completed evaluations rather than retraining the models. Rerunning this cell  would create another pair of tracking runs.

In [5]:
#Record the existing regression evaluations in MLflow.
import json

common_parameters = {
    "model_version": "v1",
    "target": "traffic_volume",
    "split_method": "chronological_unique_timestamps",
    "train_rows": 35873,
    "validation_rows": 7370,
    "test_rows": 4944,
    "original_features": 17,
    "transformed_features": 27
}

run_records = []

for model_name in ["Random Forest", "Neural Network"]:
    row = model_results.loc[
        model_results["model"] == model_name
    ].iloc[0]

    with mlflow.start_run(
        run_name=f"{model_name} v1"
    ) as run:
        mlflow.set_tags({
            "task": "Task 4 Apply an Advanced AI Technique",
            "model_family": model_name,
            "record_type": "existing_evaluation",
            "target_description": "recorded traffic volume"
        })

        mlflow.log_params(common_parameters)

        
        if model_name == "Random Forest":
            mlflow.log_params({
                parameter: forest_parameters[parameter]
                for parameter in [
                    "n_estimators",
                    "max_depth",
                    "min_samples_leaf",
                    "random_state",
                    "n_jobs"
                ]
            })

            model_artifacts = [
                "random_forest_regression_v1.joblib",
                "preparation_config_v1.joblib"
            ]

            figure_artifacts = [
                "shap_global_importance.png",
                "shap_local_waterfall.png"
            ]

        else:
            mlflow.log_params({
                "hidden_layers": json.dumps(
                    neural_config["hidden_layers"]
                ),
                "learning_rate": neural_config["learning_rate"],
                "batch_size": neural_config["batch_size"],
                "maximum_epochs": neural_config["maximum_epochs"],
                "early_stopping_patience": neural_config[
                    "early_stopping_patience"
                ],
                "epochs_completed": neural_config["epochs_completed"],
                "best_epoch": neural_config["best_epoch"],
                "random_state": neural_config["random_state"],
                "tensorflow_version": neural_config[
                    "tensorflow_version"
                ]
            })

            model_artifacts = [
                "neural_network_traffic_v1.keras",
                "neural_network_preprocessor_v1.joblib",
                "neural_network_config_v1.joblib"
            ]

            figure_artifacts = [
                "neural_network_training_curve.png"
            ]

            mlflow.log_artifact(
                str(
                    results_dir
                    / "neural_network_training_history.csv"
                ),
                artifact_path="results"
            )

        mlflow.log_metrics({
            metric: float(row[metric])
            for metric in metric_columns
        })

        for filename in model_artifacts:
            mlflow.log_artifact(
                str(models_dir / filename),
                artifact_path="models"
            )

        for filename in figure_artifacts:
            mlflow.log_artifact(
                str(figures_dir / filename),
                artifact_path="figures"
            )

        # Store the specific model's metrics as a readable artifact.
        mlflow.log_dict(
            {
                "model": model_name,
                "metrics": {
                    metric: float(row[metric])
                    for metric in metric_columns
                },
                "record_type": "existing_evaluation"
            },
            artifact_file="results/evaluation.json"
        )

        
        run_records.append({
            "model": model_name,
            "run_id": run.info.run_id
        })

    logger.info(
        "Recorded the completed MLflow run for %s.",
        model_name
    )
    

tracked_runs = pd.DataFrame(run_records)



display(tracked_runs)

2026-10-11 09:45:51,939 | INFO | __main__ | Recorded the completed MLflow run for Random Forest.
2026-10-11 09:45:52,164 | INFO | __main__ | Recorded the completed MLflow run for Neural Network.


,model,run_id
0,Random Forest,eb46daf95475495fa967b9ef5fa02e9e
1,Neural Network,a50114853d3549b98baabf39299b26d2


## Verify the tracked runs

I would read both runs back from MLflow and check that they finished successfully.

I would compare the stored metrics with the original result tables and verify that te model files and supporting artifacts are listed. 

In [6]:
# Read back the runs and verify their metrics and artifacts.

from mlflow import MlflowClient

client = MlflowClient()

expected_artifacts = {
    "Random Forest": {
        "models/random_forest_regression_v1.joblib",
        "models/preparation_config_v1.joblib",
        "figures/shap_global_importance.png",
        "figures/shap_local_waterfall.png",
        "results/evaluation.json"
    },
    "Neural Network": {
        "models/neural_network_traffic_v1.keras",
        "models/neural_network_preprocessor_v1.joblib",
        "models/neural_network_config_v1.joblib",
        "figures/neural_network_training_curve.png",
        "results/neural_network_training_history.csv",
        "results/evaluation.json"
    }
}


def list_run_files(run_id, artifact_path=""):
    files = []

    for item in client.list_artifacts(run_id, artifact_path):
        if item.is_dir:
            files.extend(list_run_files(run_id, item.path))
        else:
            files.append(item)

    return files




verified_records = []
artifact_records = []

for _, tracked in tracked_runs.iterrows():
    model_name = tracked["model"]
    run_id = tracked["run_id"]

    stored_run = client.get_run(run_id)

    assert stored_run.info.status == "FINISHED"
    assert stored_run.data.params["model_version"] == "v1"

    original = model_results.loc[
        model_results["model"] == model_name
    ].iloc[0]

    for metric in metric_columns:
        np.testing.assert_allclose(
            stored_run.data.metrics[metric],
            float(original[metric]),
            rtol=1e-10,
            atol=1e-10
        )

    stored_files = list_run_files(run_id)
    stored_paths = {item.path for item in stored_files}

    assert expected_artifacts[model_name].issubset(
        stored_paths
    ), f"Required artifacts are missing for {model_name}."

    verified_records.append({
        "model": model_name,
        "run_id": run_id,
        "status": stored_run.info.status,
        "artifact_count": len(stored_files),
        **{
            metric: stored_run.data.metrics[metric]
            for metric in metric_columns
        }
    })

    for item in stored_files:
        artifact_records.append({
            "model": model_name,
            "run_id": run_id,
            "artifact": item.path,
            "size_bytes": item.file_size
        })
        

    logger.info(
        "Verified metrics and %s listed artifacts for %s.",
        len(stored_files),
        model_name
    )

tracking_comparison = pd.DataFrame(verified_records)
artifact_inventory = pd.DataFrame(artifact_records)

tracking_comparison.to_csv(
    results_dir / "mlflow_run_comparison.csv",
    index=False
)

artifact_inventory.to_csv(
    results_dir / "mlflow_artifact_inventory.csv",
    index=False
)

logger.info("Saved the MLflow comparison and artifact inventory.")


display(tracking_comparison.round(4))


display(artifact_inventory)

2026-10-11 09:47:30,550 | INFO | __main__ | Verified metrics and 5 listed artifacts for Random Forest.
2026-10-11 09:47:30,630 | INFO | __main__ | Verified metrics and 6 listed artifacts for Neural Network.
2026-10-11 09:47:30,659 | INFO | __main__ | Saved the MLflow comparison and artifact inventory.


,model,run_id,status,artifact_count,validation_MAE,validation_R_squared,test_MAE,test_R_squared
0,Random Forest,eb46daf95475495fa967b9ef5fa02e9e,FINISHED,5,317.5692,0.9256,250.8358,0.9448
1,Neural Network,a50114853d3549b98baabf39299b26d2,FINISHED,6,316.4414,0.9307,256.9366,0.9455


,model,run_id,artifact,size_bytes
0,Random Forest,eb46daf95475495fa967b9ef5fa02e9e,figures/shap_global_importance.png,84100
1,Random Forest,eb46daf95475495fa967b9ef5fa02e9e,figures/shap_local_waterfall.png,131817
2,Random Forest,eb46daf95475495fa967b9ef5fa02e9e,models/preparation_config_v1.joblib,2409
3,Random Forest,eb46daf95475495fa967b9ef5fa02e9e,models/random_forest_regression_v1.joblib,32153120
4,Random Forest,eb46daf95475495fa967b9ef5fa02e9e,results/evaluation.json,263
5,Neural Network,a50114853d3549b98baabf39299b26d2,figures/neural_network_training_curve.png,68120
6,Neural Network,a50114853d3549b98baabf39299b26d2,models/neural_network_config_v1.joblib,1160
7,Neural Network,a50114853d3549b98baabf39299b26d2,models/neural_network_preprocessor_v1.joblib,1772
8,Neural Network,a50114853d3549b98baabf39299b26d2,models/neural_network_traffic_v1.keras,72482
9,Neural Network,a50114853d3549b98baabf39299b26d2,results/evaluation.json,265


## Value and limitations of experiment tracking

MLflow brings the model settings, evaluation metrics and supporting artifacts together under identifiable runs. This makes the existing models easier to compare and supports later deployment and monitoring work.

Random Forest had slightly lower test MAE, while the neural network had slightly higher test R squared. Their results were close.

These runs record existing evaluations rather than new training experiments. Tracking does not improve model accuracy by itself. Artifact paths are relative to the notebooks folder. Start this notebook and the MLflow viewer from that folder when reopening the project on another computer.